# 01 · Exploración de los datos OCDS

**Objetivo:** entender la estructura de un proceso de contratación publicado por el OECE en el
estándar OCDS y cómo el pipeline lo convierte en tablas.

**Entrada:** archivo OCDS (`data/sample/ocds_sample.jsonl` o un archivo de `data/raw/ocds/`).
**Salida:** ninguna; es un notebook de exploración.

> Los datos de `data/sample` son **sintéticos**. Para trabajar con datos reales descarga un año con
> `contrataciones download --year 2025` y cambia la variable `ARCHIVO` más abajo.

In [ ]:
# Configuración del entorno (ejecutar primero)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd

# Raíz del proyecto: la carpeta actual o la primera superior con pyproject.toml.
CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "pyproject.toml").is_file()), CWD)
sys.path.insert(0, str(ROOT / "src"))  # permite importar el paquete aunque no esté instalado

CURATED = ROOT / "data" / "curated"      # capa gold
PROCESSED = ROOT / "data" / "processed"  # capa silver
if not (CURATED / "fact_proceso.parquet").exists():
    raise FileNotFoundError(
        "No hay datos procesados. Ejecuta primero en la terminal: contrataciones run --source sample"
    )

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
PALETA = ["#1F2A44", "#3E6A8A", "#7FA7C2", "#C9A227", "#B5533C", "#5B8C5A", "#8C8C8C"]
plt.rcParams.update({
    "figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.prop_cycle": plt.cycler(color=PALETA),
    "axes.titleweight": "bold", "axes.titlesize": 12,
})
millones = mticker.FuncFormatter(lambda x, _: f"{x / 1e6:,.0f} M")
print("Proyecto:", ROOT)

## 1. Leer un release (un proceso de contratación)

In [ ]:
import json
from itertools import islice

from contrataciones.ingestion.ocds_source import find_ocds_files, iter_releases_from_files

reales = find_ocds_files(ROOT / "data" / "raw" / "ocds")
ARCHIVO = reales[0] if reales else ROOT / "data" / "sample" / "ocds_sample.jsonl"
print("Archivo:", ARCHIVO.name)

releases = list(islice(iter_releases_from_files([ARCHIVO]), 200))
release = releases[0]
print("Secciones del release:", list(release))
print("Campos de tender:", list(release["tender"]))

In [ ]:
# Vista resumida del JSON (las listas largas se recortan)
def recortar(obj, n=2):
    if isinstance(obj, list):
        return [recortar(x, n) for x in obj[:n]] + (["..."] if len(obj) > n else [])
    if isinstance(obj, dict):
        return {k: recortar(v, n) for k, v in obj.items()}
    return obj

print(json.dumps(recortar(release), indent=2, ensure_ascii=False)[:3000])

## 2. Del JSON anidado a tablas planas

`flatten_release` descompone el release en cinco tablas: **procesos**, **partes**, **postores**,
**adjudicaciones** y **contratos**. Es lo que hace el paso 1 del pipeline (capa *staging*).

In [ ]:
from contrataciones.processing.ocds_flattener import flatten_release

filas = flatten_release(release)
for tabla, registros in filas.items():
    print(f"{tabla:<15} {len(registros)} fila(s)")
pd.DataFrame(filas["partes"])

In [ ]:
# Las 200 primeras filas aplanadas, tabla por tabla
tablas = {}
for r in releases:
    for nombre, registros in flatten_release(r).items():
        tablas.setdefault(nombre, []).extend(registros)
tablas = {k: pd.DataFrame(v) for k, v in tablas.items()}
tablas["procesos"].head()

## 3. Particularidades de la publicación del OECE

La publicación real tiene rasgos que el pipeline maneja de forma explícita
(detalle en `docs/04_fuentes_de_datos.md`). Esta celda los mide en el archivo leído:

In [ ]:
pr, pa = tablas["procesos"], tablas["partes"]
compradores = pa[pa["roles"].fillna("").str.contains("buyer")]
proveedores_pa = pa[pa["roles"].fillna("").str.contains("tenderer|supplier")]
codigo_7 = proveedores_pa["identifier_id"].fillna("").str.fullmatch(r"\d{7}")
rasgos = pd.Series({
    "tenderPeriod con inicio = fin (plazo no informado)":
        (pr["tender_period_start"] == pr["tender_period_end"]).mean(),
    "procesos con periodo de consultas (enquiryPeriod)": pr["enquiry_period_start"].notna().mean(),
    "procesos sin estado (tender.status vacío)": pr["tender_status"].isna().mean(),
    "compradores identificados con código PE-CONSUCODE":
        compradores["identifier_scheme"].eq("PE-CONSUCODE").mean(),
    "compradores con RUC en additionalIdentifiers": compradores["additional_ruc"].notna().mean(),
    "postores/proveedores con código interno de 7 dígitos (consorcios)": codigo_7.mean(),
})
rasgos.map("{:.0%}".format).to_frame("% de filas")

Cómo se tratan en el pipeline:

* **Plazo de presentación**: si `tenderPeriod` trae inicio = fin, el plazo queda **nulo** (no 0 días)
  y se informa en el reporte de calidad; RF02 puede evaluar el periodo de consultas
  (`RF02.period: consultas` en `config/risk_rules.yaml`).
* **Entidades**: la llave es el RUC tomado de `additionalIdentifiers`; si falta, el código
  `PE-CONSUCODE`.
* **Consorcios**: no tienen RUC propio; se identifican por su código y se clasifican como
  `Consorcio` en `dim_proveedor`.
* **Estados vacíos**: una adjudicación sin estado se considera válida; un contrato sin estado se
  marca como `sin_estado`.

## 4. La capa silver (datos limpios del pipeline)

In [ ]:
procesos = pd.read_parquet(PROCESSED / "procesos.parquet")
print(procesos.shape)
procesos.info()

In [ ]:
resumen = pd.DataFrame({
    "procesos": procesos.groupby("metodo_detalle").size(),
    "valor_referencial_mediano": procesos.groupby("metodo_detalle")["valor_referencial_pen"].median(),
}).sort_values("procesos", ascending=False)
resumen

In [ ]:
procesos["categoria"].value_counts().plot.barh(title="Procesos por categoría")
plt.xlabel("N.° de procesos"); plt.ylabel(""); plt.show()

## Conclusiones y siguientes pasos

* Cada línea del archivo es un proceso completo (*release compilado*).
* Los montos están en `tender.value` (valor referencial) y `awards[].value` (adjudicado).
* Las organizaciones aparecen una vez en `parties` con sus **roles** (comprador, postor, proveedor).
* Siguiente notebook: **02 · Calidad de datos**.